[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MLManuals/Deep-Learning/blob/main/chapter12_datasets_dataloaders_and_batching/notebook.ipynb)

# Chapter 12: Datasets, DataLoaders and Batching

Run the cell below once, first, to install any missing packages (only needed on Google Colab; skip it if you're running locally and already have them).

In [ ]:
%pip install -q torch numpy scikit-learn

Chapter 11 quietly got away with something. All 105 training flowers went through the network at once, in a single line, because 105 rows of four numbers is nothing at all. The photographs in Book 7 will not be so obliging. A modest collection of a hundred thousand colour images is tens of gigabytes, which will not fit in the memory of any ordinary machine, let alone on a graphics card.

The way out is older than neural networks. In 1951 two statisticians, Herbert Robbins and Sutton Monro, published a method for solving problems where you cannot measure a quantity exactly but can take noisy samples of it. Their idea was that you do not need the exact answer at every step. A rough estimate is enough, as long as it points the right way on average and you take enough steps. That field is called stochastic approximation, and the word **stochastic** simply means involving randomness, which answers the question Chapter 10 left hanging about what the S in SGD stands for.

Applied to training, the idea is this. Instead of computing the loss over every example before taking one step, compute it over a small random handful and step immediately. Each step is based on a rough estimate of the true slope, but you get far more steps, and the roughness turns out to help rather than hurt. Before you read on, try to guess the answers to these four questions.

1. **If your data does not fit in memory, what is the obvious thing to do about it?**
2. **In one pass over the data, how many weight updates does a small batch give you compared with one large one?**
3. **Why might it matter in what order the examples arrive?**
4. **Could a rougher, noisier estimate of the slope ever be better than the exact one?**

## Why Feed Data in Batches

A **batch** is a small group of examples handled together, and a **mini-batch** is the usual name when that group is much smaller than the whole dataset. Three separate arguments push in the same direction, and they are worth keeping apart because they are often muddled together.

The first is memory, which is the one everybody meets. If the data will not fit, you have no choice. The second is speed of learning, and it is less obvious. Chapter 11 made exactly one weight update per pass through the flowers. Splitting those 105 flowers into batches of 8 gives 14 updates per pass instead of one, so the network improves fourteen times as often for roughly the same arithmetic. The third is the noise itself, which Chapter 6 already promised would come back.

Figure 12.1 shows one epoch under both arrangements.

> 🖼️ **Figure 12.1:** One pass through the same sixteen examples. Processed whole it yields a single update, while split into four batches it yields four, each one slightly rougher.

**Batching exists first because large data will not fit in memory, but its everyday benefit is that you get many more weight updates out of the same pass through the data.**

## The Dataset and the DataLoader

PyTorch splits the job in two. A **Dataset** knows how to produce one example when asked for it by number, and a **DataLoader** knows how to group examples into batches, shuffle them and hand them over one batch at a time. When your data is already in tensors, `TensorDataset` is the dataset you want.

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split

iris = load_iris()
X_train, X_test, y_train, y_test = train_test_split(
    iris.data, iris.target, test_size=0.3,
    random_state=7, stratify=iris.target)
mean, std = X_train.mean(axis=0), X_train.std(axis=0)

In [ ]:
Xtr = torch.tensor((X_train - mean) / std, dtype=torch.float32)
Xte = torch.tensor((X_test - mean) / std, dtype=torch.float32)
ytr = torch.tensor(y_train)
yte = torch.tensor(y_test)

train_ds = TensorDataset(Xtr, ytr)
print(len(train_ds))
print(train_ds[0][0].numpy().round(2), train_ds[0][1].item())

The dataset reports its length and hands back one flower at a time, as a pair holding the four measurements and the species. That is all a dataset has to do, and when you reach real images in Book 7 the only thing that changes is that producing an example means reading a file from disk.

The loader wraps it and does the grouping.

In [ ]:
loader = DataLoader(train_ds, batch_size=16, shuffle=True)
print(len(loader))

Seven batches, because 105 flowers in groups of 16 is six full batches with nine left over. That last short batch is normal and PyTorch simply hands it to you, which matters when you average anything per batch. Looping over the loader gives the batches in turn.

In [ ]:
torch.manual_seed(7)
for step, (xb, yb) in enumerate(loader):
    print(step, tuple(xb.shape), yb[:5].numpy())
    if step == 1:
        break

Each batch is a tensor of 16 flowers by 4 measurements, which slots straight into the model from Chapter 11 with no change at all, because Chapter 8's broadcasting already handles a whole batch at once.

## Shuffling, Epochs and Steps

Three words get used constantly and are worth pinning down. An **epoch** is one complete pass through the training data. A **step**, sometimes called an iteration, is one weight update, meaning one batch. The number of steps in an epoch is the dataset size divided by the batch size, rounded up.

In [ ]:
import math

for bs in [1, 8, 16, 32, 105]:
    print(bs, math.ceil(105 / bs), len(DataLoader(train_ds, batch_size=bs)))

The arithmetic and PyTorch agree. Notice the last row, which is Chapter 11's arrangement, where one epoch was one step. Table 12.1 fixes the vocabulary.

**Table 12.1: The three words that describe any training run, with Chapter 11's flowers as the example.**

| Word | What it means | For 105 flowers in batches of 8 |
|---|---|---|
| Batch size | Examples handled together | 8 |
| Step | One weight update, one batch | 14 of them per epoch |
| Epoch | One full pass through the data | 1 |

The `shuffle=True` argument reorders the data before every epoch, and leaving it out is a real mistake rather than a stylistic one. The iris data arrives sorted by species, so without shuffling the first batches would be entirely setosa, the middle ones entirely versicolor and the last ones entirely virginica. The network would spend each epoch being pulled hard towards one species and then another, and it would learn badly or not at all. **Always shuffle the training data, because a batch that contains only one kind of example gives a slope that points somewhere no sensible step would go.** Validation and test loaders need no shuffling, since nothing is learned from them and the order changes nothing.

## Stochastic versus Full Batch

With the loader in place, the training loop changes by one line. The five steps from Chapter 11 now sit inside a loop over batches, and everything else is identical.

In [ ]:
loss_fn = nn.CrossEntropyLoss()

def train(batch_size, epochs=30, lr=0.1):
    torch.manual_seed(7)
    model = nn.Sequential(nn.Linear(4, 16), nn.ReLU(), nn.Linear(16, 3))
    optimizer = torch.optim.SGD(model.parameters(), lr=lr)
    loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)

    updates = 0
    for epoch in range(epochs):
        for xb, yb in loader:
            loss = loss_fn(model(xb), yb)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            updates += 1
    return model, updates

The two extremes have names. Using the whole dataset in one go, as Chapter 11 did, is **full batch** or batch gradient descent. Using exactly one example per step is **stochastic gradient descent** in its original sense, and it is where the name comes from. Everything in between is mini-batch, which is what people actually use and what the name SGD now loosely refers to.

The next cell trains with a single flower per step and watches the loss on each individual batch.

In [ ]:
torch.manual_seed(7)
tiny = nn.Sequential(nn.Linear(4, 16), nn.ReLU(), nn.Linear(16, 3))
opt = torch.optim.SGD(tiny.parameters(), lr=0.1)
one_at_a_time = DataLoader(train_ds, batch_size=1, shuffle=True)

Each pass of the loop below handles a single flower, so one epoch is 105 separate updates.

In [ ]:
seen = []
for xb, yb in one_at_a_time:
    loss = loss_fn(tiny(xb), yb)
    seen.append(round(loss.item(), 2))
    opt.zero_grad()
    loss.backward()
    opt.step()
print(seen[:8])

Those numbers jump around rather than falling steadily, and nothing is wrong. Each one is the loss on a single flower, so an easy setosa scores low and a borderline virginica scores high, regardless of how well the network is doing overall. Figure 12.2 plots two epochs of them.

> 🖼️ **Figure 12.2:** The loss on each single flower while training one at a time. The jitter is the randomness that gives stochastic gradient descent its name.

This is the jitter Chapter 6 promised. A run that settles into a shallow dip in the landscape gets shaken by the next awkward example, and that shaking is often enough to knock it back out. The noise that looks like a defect is doing useful work, which answers the fourth opening question.

## How Batch Size Changes Training

Batch size is a genuine setting to choose, not a technicality. The next cell trains the same network for the same 30 epochs with four different batch sizes and reports how many updates each one made and where it ended up.

In [ ]:
for bs in [1, 8, 32, 105]:
    model, updates = train(bs)
    with torch.no_grad():
        train_loss = loss_fn(model(Xtr), ytr).item()
        acc = (model(Xte).argmax(1) == yte).float().mean().item()
    print(bs, updates, round(train_loss, 4), round(acc, 3))

The pattern is unmistakable. Thirty epochs is thirty epochs in every row, so each one saw the data the same number of times, but the number of updates falls from 3150 to 30 and the final loss rises right along with it. Chapter 11 needed 301 full batch epochs to reach a loss of 0.0987, and batches of 8 get past that in 30. Figure 12.3 shows three of these runs.

> 🖼️ **Figure 12.3:** The same thirty epochs at three batch sizes. Smaller batches make more updates per epoch and therefore travel much further in the same number of passes.

It would be wrong to read that table as saying smaller is simply better. Each row counts epochs, not time, and a batch of 1 does 3150 separate small pieces of arithmetic where a batch of 105 does 30 large ones. Large batches use hardware far more efficiently, especially a GPU, which is built to do one big calculation faster than many tiny ones. The real choice balances three things, which Table 12.2 sets out.

**Table 12.2: The trade in choosing a batch size. Most practitioners start at 32 or 64 and only move when memory or speed forces them to.**

| Batch size | What you gain | What you give up |
|---|---|---|
| Small, 1 to 16 | Many updates, helpful noise, little memory | Slow per epoch, hardware poorly used |
| Medium, 32 to 256 | The usual balance for most work | Nothing in particular, which is why it is usual |
| Large, 512 and up | Fast per epoch, uses a GPU well | Few updates, more memory, less helpful noise |

One warning saves a lot of confusion later. Batch size and learning rate pull on each other, because a larger batch gives a smoother, more reliable slope, which can safely carry a larger step. Changing one and keeping the other fixed often makes training worse for reasons that look mysterious. **Start with a batch size of 32 or 64, move it only when memory or speed makes you, and expect to revisit the learning rate whenever you do.**

## Practice and Review

Reading about deep learning is only half of learning it. Pulling an idea back out of your memory makes it stick much better than reading it again, so this section gives you two kinds of practice, one for your hands and one for your memory.

### Try It Yourself

1. **Turn off shuffling.** Build the loader with `shuffle=False`, train with batches of 8 and compare the final loss with 0.0772. Print the labels in the first batch to see why it went wrong.
2. **Count the short batch.** Use a batch size of 16 and print `xb.shape` for every batch of one epoch. Find the short one and explain its size from 105 and 16.
3. **Match the budget.** Train with a batch of 105 for 300 epochs rather than 30, so it makes 300 updates, and compare its loss with the batch of 8 after 30 epochs, which made 420.

### Check Your Understanding

Try to answer each question from memory before you look at the answers that follow.

1. **What does the word stochastic mean, and where does it come from here?**
2. **What is the difference between an epoch and a step?**
3. **Why must the training data be shuffled?**
4. **Why did a batch of 8 reach a lower loss than a batch of 105 in the same 30 epochs?**
5. **Why is a batch of 1 not simply the best choice?**

### Answers

1. It means involving randomness, and it refers to estimating the slope from a random handful of examples rather than from all of them.
2. An epoch is one full pass through the data, while a step is one weight update, meaning one batch.
3. Because the data may be sorted, so unshuffled batches can contain a single class and give a slope that points somewhere useless.
4. Because it made 420 updates in those epochs instead of 30, so it took far more steps downhill.
5. Because it uses hardware badly and is slow per epoch, and the extra noise stops being helpful once there is too much of it.

## What Comes Next

You can now wrap data in a dataset, serve it in shuffled batches, say exactly what an epoch and a step are, and choose a batch size knowing what you are trading.

- **Chapter 13, Optimizers: SGD, Momentum and Adam,** replaces the plain downhill step with the ones people actually use.
- **Chapter 14, Learning Rates and Schedules,** tackles the setting that Chapter 6 called the one dial that matters.

Chapter 13 starts with a shape that defeats plain gradient descent, a valley that is steep across and almost flat along, where every step bounces off the walls while barely moving forward.